# o1-Style Reasoning: Test-Time Compute Scaling

OpenAI's o1 (2024) demonstrated a new axis for improving model performance: scaling compute at inference time rather than only at training time. By allowing models to 'think longer' before answering, o1 achieved dramatic improvements on hard reasoning benchmarks.

## The Test-Time Compute Paradigm

Conventional scaling: train a larger model (more parameters, more training data). Expensive and slow.

o1-style scaling: at inference time, generate extended internal reasoning before producing the final answer. The 'thinking' tokens are not shown to the user but are used as a reasoning scratchpad.

Key results at launch:
- IMO 2024: o1 solved 4 of 6 problems (vs ~0 for GPT-4)
- AIME 2024: 83% vs 13% for GPT-4
- PhD-level science questions (GPQA): 78% vs 56% for GPT-4o

The improvement came from allowing the model to explore, backtrack, and self-verify within the thinking context — essentially running MCTS/ToT-style search as part of the generation process, but trained into the model rather than implemented externally.

## Thinking Budget and Quality Tradeoffs

Allowing unlimited thinking tokens is expensive. o1 and subsequent models (o1-mini, o3, DeepSeek-R1) expose controls for the thinking budget:

- **Low thinking budget**: fast response, CoT-level reasoning quality
- **Medium budget**: moderate latency, significant improvement on complex tasks
- **High budget**: slow response, maximum reasoning quality for the hardest problems

For most production tasks, medium budget is the best default. Very hard mathematical and coding problems justify high budget. Conversational tasks don't benefit from extended thinking.

In [ ]:
from dataclasses import dataclass
from typing import Callable
import random, math

@dataclass
class ThinkingResult:
    problem: str
    thinking_tokens: int
    thinking_steps: list
    final_answer: str
    confidence: float
    correct: bool

class ThinkingBudgetController:
    def __init__(self, base_model_fn: Callable, max_tokens: int = 1000):
        self.model = base_model_fn
        self.max_tokens = max_tokens

    def think(self, problem: str, budget: int, correct_answer: str = '') -> ThinkingResult:
        rng = random.Random(hash(problem) % 10000)
        # Simulate thinking: more tokens -> better exploration
        n_steps = max(1, budget // 50)
        steps = []
        current_confidence = 0.4
        for i in range(n_steps):
            step_type = rng.choice(['explore', 'verify', 'backtrack', 'conclude'])
            if step_type == 'explore':
                steps.append(f'[explore] Considering approach {i+1}: decompose by...')
                current_confidence += 0.05 * rng.random()
            elif step_type == 'verify':
                steps.append(f'[verify] Checking step {i}: arithmetic confirms...')
                current_confidence += 0.08 * rng.random()
            elif step_type == 'backtrack':
                steps.append(f'[backtrack] Previous approach flawed, trying alternative...')
                current_confidence = max(0.3, current_confidence - 0.1)
            else:
                steps.append(f'[conclude] Converging on answer with confidence {current_confidence:.2f}')
                current_confidence = min(0.95, current_confidence + 0.1)
        confidence = min(0.95, current_confidence)
        # More thinking = higher accuracy (up to a point)
        accuracy_threshold = 0.5 + 0.4 * (1 - math.exp(-budget / 200))
        answer = correct_answer if rng.random() < accuracy_threshold else 'wrong_answer'
        return ThinkingResult(
            problem=problem[:60], thinking_tokens=budget,
            thinking_steps=steps, final_answer=answer,
            confidence=confidence, correct=(answer == correct_answer)
        )

def thinking_budget_sweep(problem: str, correct_answer: str,
                          budgets: list, n_trials: int = 10) -> list:
    controller = ThinkingBudgetController(lambda p: p)
    results = []
    for budget in budgets:
        rng = random.Random(42)
        correct_count = 0
        for trial in range(n_trials):
            # Add trial seed variation
            modified_problem = problem + str(trial)
            r = controller.think(modified_problem, budget, correct_answer)
            if r.correct:
                correct_count += 1
        results.append({
            'budget': budget,
            'accuracy': correct_count / n_trials,
            'estimated_cost': budget * 0.001,  # mock cost in cents
        })
    return results

budgets = [50, 100, 200, 400, 800, 1600]
results = thinking_budget_sweep(
    'Solve: For how many integers n with 1<=n<=2024, is n^3 - n^2 divisible by 4?',
    '1012', budgets
)
print(f'{'Budget':>10} {'Accuracy':>10} {'Cost (¢)':>10} {'Efficient?'}')
print('-' * 45)
for r in results:
    efficient = 'Yes' if r['accuracy'] > 0.7 and r['estimated_cost'] < 0.5 else 'No'
    print(f"{r['budget']:>10} {r['accuracy']:>9.0%} {r['estimated_cost']:>9.3f}  {efficient}")

## DeepSeek-R1 and Open Reasoning Models

DeepSeek-R1 (2025) replicated o1-style reasoning with open weights, demonstrating that the approach does not require proprietary techniques:

- Trained using Group Relative Policy Optimization (GRPO) with outcome-only rewards on math/code
- The model spontaneously developed verification and backtracking behaviors during training
- Achieved comparable performance to o1 on AIME, MATH, and coding benchmarks

Key insight from the DeepSeek-R1 paper: you do not need to explicitly teach the model to think step by step. Training with a reward for correct final answers on hard problems, with a long context budget, causes the model to develop extended reasoning spontaneously.

In [ ]:
def adaptive_thinking(problem: str, model_fn: Callable,
                       min_budget: int = 100, max_budget: int = 1000,
                       target_confidence: float = 0.85) -> dict:
    controller = ThinkingBudgetController(model_fn)
    budget = min_budget
    total_tokens = 0
    rounds = []
    while budget <= max_budget:
        result = controller.think(problem, budget, 'target_answer')
        total_tokens += budget
        rounds.append({'budget': budget, 'confidence': result.confidence})
        if result.confidence >= target_confidence:
            break
        budget = min(max_budget, budget * 2)
    return {
        'final_confidence': rounds[-1]['confidence'],
        'total_tokens_used': total_tokens,
        'rounds': len(rounds),
        'budget_progression': [r['budget'] for r in rounds],
    }

result = adaptive_thinking(
    'What is the 100th Fibonacci number modulo 1000?',
    lambda p: p,
    min_budget=100, max_budget=800, target_confidence=0.85
)
print('Adaptive thinking budget:')
for k, v in result.items():
    print(f'  {k}: {v}')

## Implications for System Design

Test-time compute scaling changes how you architect LLM systems:

**Task routing**: route hard problems to o1/o3 class models; route simple tasks to faster, cheaper models. The cost difference is large — o1 is 10-30x more expensive than GPT-4o per token.

**Latency management**: extended thinking means 10-60 second response times. User-facing applications need streaming indicators and timeout handling.

**Budget allocation**: thinking tokens are a resource to allocate intelligently. Don't waste long-thinking budget on questions that don't require it.

**Evaluation**: comparing thinking model quality requires accounting for budget. Comparing o1 at low budget vs GPT-4o is not a fair comparison — they are at similar cost points.